# 机制分析  
将证券划为可融资融券 vs 不可融资融券，类似异质性 

## 导入库

In [44]:
import os
import dotenv
import re
import warnings
import polars as pl
import plotly  
import plotly.express as px
from plotly.subplots import make_subplots
import statsmodels.api as sm
dotenv.load_dotenv()

True

## 超参数

In [45]:
# 基本配置
BASELINE_TASK_ID_PREFIX = 'baseline1'  # 基线任务id前缀
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{BASELINE_TASK_ID_PREFIX}' # 保存基本路
SAVE = True # 是否保存数据

# 数据库
CONNECTION_URL = os.getenv("POSTGRES_URL")
ENGINE = "adbc"

# 异质性组
FIN_GROUP = 0 # 0表示不能融资融券，1表示可以


## 读取数据

In [46]:
ma_df = pl.read_parquet(SAVE_BASE_DIR + '/MA因子_copy.parquet')
ma_df.head()

date,portfolio,MA,return
date,str,f64,f64
2023-01-01,"""301019""",0.673006,0.0132
2022-11-01,"""002010""",0.811182,-0.069629
2017-03-01,"""600770""",0.607574,-0.1574
2024-03-01,"""300512""",0.839616,-0.0327
2006-11-01,"""002031""",0.649186,0.0552


In [47]:
margin_and_short = pl.read_database_uri(
    uri=CONNECTION_URL,
    query='''SELECT *
    FROM com_info.margin_and_short''',
    engine=ENGINE,
).rename({'stkcd':'portfolio'})

## 连接筛选数据

In [48]:
ma_df

date,portfolio,MA,return
date,str,f64,f64
2023-01-01,"""301019""",0.673006,0.0132
2022-11-01,"""002010""",0.811182,-0.069629
2017-03-01,"""600770""",0.607574,-0.1574
2024-03-01,"""300512""",0.839616,-0.0327
2006-11-01,"""002031""",0.649186,0.0552
…,…,…,…
2024-05-01,"""301578""",0.863275,-0.2731
2019-03-01,"""002557""",0.637052,-0.059342
2024-01-01,"""603267""",0.652789,0.023941


In [49]:
ma_df = ma_df.join(margin_and_short, on='portfolio',how='left')
ma_df = ma_df.with_columns(pl.col('finance_and_short').fill_null(0).alias('finance_and_short'))
ma_df = ma_df.filter(pl.col('finance_and_short') == FIN_GROUP)

In [50]:
ma_df.write_parquet(SAVE_BASE_DIR + '/MA因子.parquet')